# Agentic RAG: Router-Retriever System with PDF and Web Search Tools

**Overview**
This project challenges you to design and implement an Agentic Retrieval-Augmented Generation (RAG) system that employs multiple role-based AI agents to process user queries. The workflow includes a Router Agent that classifies questions and a Retriever Agent that executes the most suitable retrieval method, using either a PDF-based vector search or a web search tool. The system demonstrates intelligent orchestration across these agents, producing accurate, source-grounded answers.

**Instructions**
- Review the learning modules and any linked documentation for CrewAI and the tools used
- Read each section (situation, tasks, actions, and result) to understand the deliverables and workflow expectations thoroughly
- Implement the complete solution using Python in Jupyter notebook, with appropriate orchestration and logging tools
- Submit your final working notebook and a brief README file through the LMS
- Ensure all project elements, from role definitions to reasoning trace visualizations, are included and functional

**Situation**
As a developer, you are tasked with building an AI-powered multi-agent system to answer domain-specific questions using both static and dynamic sources. The system comprises:
- Router agent: It decides the retrieval path based on the question
- Retriever agent: It executes the retrieval from the chosen source and formulates the answer
Available tools include a PDF search tool for static, domain-specific content and a Web search tool for retrieving fresh information from the internet. An optional generation path directly uses the LLM without retrieval.

This notebook implements the assignment using CrewAI's hierarchical process: the Router agent acts as the crew's `manager_agent`, classifying each question as PDF / WEB / DIRECT and delegating to the Retriever agent, which holds both tools and formulates the final answer.

Install dependencies (LangChain + FAISS for the PDF pipeline, Tavily for web search, CrewAI for orchestration). Copy `.env.example` to `.env` and fill in `OPENAI_API_KEY`, `OPENAI_MODEL_NAME`, and `TAVILY_API_KEY` before running.

In [ ]:
%pip install langchain langchain-openai langchain-community 
%pip install langchain-text-splitters langchain-classic langchain-core faiss-cpu pypdf
%pip install langchain-tavily tavily-python 
%pip install crewai
%pip install litellm
%pip install pydantic dotenv

Load environment variables and fail fast if a required API key is missing.

In [1]:
# Import environment variables
import os
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

# Check if API keys were loaded from environment variables
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY environment variable not found!")
    
if not os.getenv("TAVILY_API_KEY"):
    raise ValueError("TAVILY_API_KEY environment variable not found!")


`TavilySearchTool` posts the query to Tavily's API and returns the top 3 results as raw `"title - url"` lines for the agent to reason over.

In [2]:
# Build web search tool using tavily
import requests
from crewai.tools import BaseTool

class TavilySearchTool(BaseTool):
    name: str = "Web Search"
    description: str = "Search the web for recent information"

    def _run(self, query: str):
        url = "https://api.tavily.com/search"

        payload = {
            "api_key": os.getenv("TAVILY_API_KEY"),
            "query": query,
            "max_results": 3
        }

        response = requests.post(url, json=payload)
        data = response.json()
        
        results = []
        for r in data["results"]:
                results.append(f"{r['title']} - {r['url']}")
        
        return "\n".join(results)

search_tool = TavilySearchTool()

Load the PDF, split it into 1000-character chunks (50-character overlap), embed the chunks with OpenAI embeddings, and index them in an in-memory FAISS vectorstore.

In [3]:
# Split the PDF and create a vector store for retrieval
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

# Load PDF document
loader = PyPDFLoader(
    file_path="input/AI-interview-questions-2026.pdf",
    extraction_mode="layout"
)
pdf_document=loader.load()
print(f"Loaded {len(pdf_document)} pages")

# Split the document into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=50
)
pdf_chunks=text_splitter.split_documents(pdf_document)
print(f"Split into {len(pdf_chunks)} chunks")

# Generate embeddings for the chunks
embeddings = OpenAIEmbeddings(
    model="text-embedding-ada-002",
    api_key=os.getenv("OPENAI_API_KEY")
)
vectorstore = FAISS.from_documents(pdf_chunks, embeddings)

C:\Users\rohit\AppData\Local\Temp\ipykernel_27380\3807880452.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 19 pages
Split into 46 chunks


`PDFSearchTool` wraps `vectorstore.similarity_search` and returns raw snippets rather than a synthesized answer, so the Retriever agent — not a hidden `RetrievalQA` chain — formulates the final response.

In [5]:
# Build PDF RAG search tool
# Returns raw retrieved snippets (like TavilySearchTool returns raw titles/URLs)
# so the Retriever agent formulates the final answer itself, keeping its
# reasoning visible in the trace instead of hiding it behind a second LLM call.

from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

class PDFSearchTool(BaseTool):
    name: str = "PDF Search"
    description: str = "Search the AI interview questions PDF for domain-specific reference content."

    def _run(self, query: str):
        docs = vectorstore.similarity_search(query, k=3)
        results = []
        for d in docs:
            page = d.metadata.get("page", "unknown")
            snippet = d.page_content.strip().replace("\n", " ")[:500]
            results.append(f"[Page {page}] {snippet}")
        return "\n\n".join(results)

pdf_tool = PDFSearchTool()

### Router and Retriever agents (hierarchical Crew)

The Router is the crew's `manager_agent` and must not also appear in `agents=[...]`. Assigning the one `Task` to `agent=retriever` restricts the Router's delegation to that single coworker, so it can only delegate — never call a tool itself. The Retriever holds both tools (`pdf_tool`, `search_tool`) and follows whichever instruction the Router delegates with, including calling no tool for `DIRECT` questions.

Trace logging combines `verbose=True`, `output_log_file="trace.log"`, `tracing=True` (CrewAI's hosted trace viewer), a `Task.callback`, and per-agent token usage summaries.

In [6]:
from crewai.llm import LLM
from crewai import Agent, Task, Crew, Process

router_llm = LLM(
    model=os.getenv("OPENAI_MODEL_NAME"),
    api_key=os.getenv("OPENAI_API_KEY"),
    is_litellm=True,
    temperature=0
)
retriever_llm = LLM(
    model=os.getenv("OPENAI_MODEL_NAME"),
    api_key=os.getenv("OPENAI_API_KEY"),
    is_litellm=True,
    temperature=0.3
)

def log_task(output):
    print(f"\n--- TASK COMPLETE ---\n{output.raw}\n")

async def run_agentic_rag(user_question: str):
    router = Agent(
        role="Query Router",
        goal="Classify the question as PDF, WEB, or DIRECT, then delegate it to the Answer Retriever "
             "with clear instructions on which single tool (if any) to use.",
        backstory=("A triage analyst who knows the AI-interview-questions-2026.pdf reference document's "
                   "scope and decides whether a question is answerable from it, needs fresh web info, "
                   "or needs no retrieval at all."),
        llm=router_llm,
        verbose=True,
        allow_delegation=True,
    )

    retriever = Agent(
        role="Answer Retriever",
        goal="Follow the Router's instructions, use at most one retrieval tool if told to, "
             "and formulate a grounded final answer.",
        backstory=("A careful research assistant that only uses the source it's instructed to use "
                   "and never fabricates facts beyond what it retrieves."),
        llm=retriever_llm,
        tools=[pdf_tool, search_tool],
        verbose=True,
    )

    task_answer = Task(
        description=(
            f"Answer this question: {user_question}\n\n"
            "First classify it as PDF (answerable from the static AI-interview-questions-2026.pdf "
            "reference), WEB (needs fresh/current info), or DIRECT (general knowledge, no retrieval). "
            "Then delegate to the Answer Retriever, telling it exactly which single tool to use "
            "(PDF Search, Web Search, or none) and have it produce the final answer."
        ),
        expected_output="A clear final answer that states which source (PDF, web, or general knowledge) was used.",
        agent=retriever,
        callback=log_task,
    )

    crew = Crew(
        agents=[retriever],          # manager_agent must NOT be in this list
        tasks=[task_answer],
        process=Process.hierarchical,
        manager_agent=router,
        verbose=True,
        output_log_file="trace.log",
        tracing=True,
    )

    # Jupyter runs its own asyncio event loop, so a synchronous crew.kickoff()
    # raises "invoked synchronously from within a running event loop" here.
    result = await crew.kickoff_async()
    print(f"\n=== Question: {user_question} ===\n{result.raw}\n")
    return {
        "question": user_question,
        "answer": result.raw,
        "crew_token_usage": result.token_usage,
        "router_token_usage": router_llm.get_token_usage_summary(),
        "retriever_token_usage": retriever_llm.get_token_usage_summary(),
    }

Run three sample questions covering all three routes (PDF, WEB, DIRECT). Uses `kickoff_async()` since Jupyter's own event loop doesn't allow the synchronous `kickoff()`.

In [7]:
# Run sample questions covering all three routing paths
sample_questions = [
    "What are common AI interview questions about overfitting?",  # PDF
    "What AI hiring trends are being discussed in 2026?",         # WEB
    "What is 2 + 2?",                                             # DIRECT
]

runs = []
for q in sample_questions:
    runs.append(await run_agentic_rag(q))

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: f5076bcf-9ba9-4dc2-88aa-328939e7b0e8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Answer this question: What are common AI interview questions about overfitting?                          │
│                                                                                                                 │
│  First classify it as PDF (answerable from the static AI-interview-questions-2026.pdf reference), WEB (needs    │
│  fresh/current info), or DIRECT (general knowledge, no retrieval). Then delegate to the Answer Retriever,       │
│  telling it exactly which single tool to use (PDF Search, Web Search, or none) and have it produce the final    │
│  answer.                                                                                                        │
│  ID: 5a5f8ef4-7285-4c85-8bff-7a4ea31e317a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│  Task: Answer this question: What are common AI interview questions about overfitting?                          │
│                                                                                                                 │
│  First classify it as PDF (answerable from the static AI-interview-questions-2026.pdf reference), WEB (needs    │
│  fresh/current info), or DIRECT (general knowledge, no retrieval). Then delegate to the Answer Retriever,       │
│  telling it exactly which single tool to use (PDF Search, Web Search, or none) and have it produce the final    │
│  answer.                                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: pdf_search                                                                                               │
│  Args: {'query': 'common AI interview questions about overfitting'}                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool pdf_search executed with result: [Page 16] This question addresses the ethical and practical issues of LLM-generated content. A strong answer would demonstrate awareness of these problems and discuss techniques like the following.  H...

...


--- TASK COMPLETE ---
The answer to the question 'What is 2 + 2?' is 4. This answer is based on general knowledge and does not require any retrieval from the PDF or web.



╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Answer this question: What is 2 + 2?                                                                     │
│                                                                                                                 │
│  First classify it as PDF (answerable from the static AI-interview-questions-2026.pdf reference), WEB (needs    │
│  fresh/current info), or DIRECT (general knowledge, no retrieval). Then delegate to the Answer Retriever,       │
│  telling it exactly which single tool to use (PDF Search, Web Search, or none) and have it produce the final    │
│  answer.                                                                                                        │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: a51cb7cd-d152-4c0b-b12e-e0545df3f69b                                                                       │
│  Final Output: The answer to the question 'What is 2 + 2?' is 4. This answer is based on general knowledge and  │
│  does not require any retrieval from the PDF or web.                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


=== Question: What is 2 + 2? ===
The answer to the question 'What is 2 + 2?' is 4. This answer is based on general knowledge and does not require any retrieval from the PDF or web.



╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Print each run's question, answer, and token usage side by side.

In [ ]:
# Summary table tying question -> answer -> token cost together
for run in runs:
    print(f"Question: {run['question']}")
    print(f"Answer: {run['answer'][:200]}")
    print(f"Crew tokens: {run['crew_token_usage']}")
    print(f"Router tokens: {run['router_token_usage']}")
    print(f"Retriever tokens: {run['retriever_token_usage']}")
    print("-" * 80)